# German Industry Run-Ups, Crashes, and Post-Run-Up Performance

This notebook presents the German empirical analysis based on the framework of Greenwood, Shleifer, and You (2019), *Bubbles for Fama*. It is a Germany-focused adaptation, not a claim of an identical replication of the original U.S. sample.

The calculations were executed in Stata using `Untitled_revised`. This notebook documents the complete workflow, displays the executed exhibits, and interprets the German results. It does not recalculate the raw-data pipeline in Python.

## 1. Research question and empirical objectives

The main question is whether unusually large price increases in German industries are followed by elevated crash risk and distinctive post-run-up performance.

The analysis has five objectives:

1. Describe German industry episodes that satisfy a large prior price increase.
2. Examine subsequent returns and maximum drawdowns.
3. Compare crash probabilities after strict 100% run-ups with broader observations.
4. Study whether the run-up indicator predicts crashes in simple regressions, with and without a volatility control.
5. Visualize industry and market performance around the identified 100% run-up episodes.

The German results are interpreted against the broad patterns reported in the U.S. and international evidence, while recognizing that the German sample is smaller and has different industry composition and data coverage.

## 2. Input data and files that are not used

The Stata workflow uses `comp_global_daily.dta` as the source dataset. It filters this file to German common-equity securities and uses the security-level variables needed to construct industry portfolios.

Files such as `funda.dta`, `msci_rf.dta`, and `gdp.dta` are not read by the executed workflow. They are therefore not part of the calculations reported here. No risk-free return series is introduced, so risk-free-return fields are not reported as if they were observed.

The final exhibits are generated in Stata and read below from Excel or PNG files.

## 3. Portable project setup

The notebook searches the current directory, nearby parent directories, and their immediate child folders for the final exhibit files. This avoids hard-coding the local Windows path used by Stata.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

FINAL_FILES = [
    'germany_table1_paper_style.xlsx',
    'germany_event_time_paths.png',
    'germany_event_time_paths.xlsx',
    'germany_table3_main.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_crash_test.xlsx',
]

def find_project_dir():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    for base in [cwd, *cwd.parents]:
        try:
            candidates.extend(item for item in base.iterdir() if item.is_dir())
        except OSError:
            pass
    scored = []
    for candidate in dict.fromkeys(candidates):
        if candidate.is_dir():
            present = sum((candidate / filename).exists() for filename in FINAL_FILES)
            if present > 0:
                scored.append((present, candidate))
    if not scored:
        raise FileNotFoundError('Could not find the folder containing the German exhibit files.')
    return sorted(scored, key=lambda item: (-item[0], str(item[1])))[0][1]

PROJECT_DIR = find_project_dir()
display(Markdown(f'**Project folder:** `{PROJECT_DIR}`'))
for filename in FINAL_FILES:
    print(f'{filename}: {"FOUND" if (PROJECT_DIR / filename).exists() else "MISSING"}')

def read_excel(filename):
    path = PROJECT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing required exhibit: {filename}')
    return pd.read_excel(path)


## 4. Security sample and currency normalization

The Stata workflow keeps observations with `fic == "DEU"`, keeps common equity using `tpci == "0"`, requires a valid `gsector` classification, and requires a positive price. Security identifiers are represented by `gvkey` and `iid`.

The adjusted total-return price is constructed as:

`(prccd × fx_eur / ajexdi) × trfd`

EUR observations use a conversion factor of one. DEM observations use the fixed conversion factor `1 / 1.95583`. This step is important for Germany because an unhandled currency transition could appear as a false market-wide price movement. The revised Stata workflow records currency changes as a diagnostic.

## 5. Daily return cleaning and monthly security returns

Daily returns are constructed from adjusted total-return prices. The workflow flags likely reversal errors when a very large current or previous return is followed by a small two-day compounded return. It also flags extreme current returns above 200%. Flagged daily returns are set to missing rather than being treated as genuine economic movements.

For monthly returns, the last available trading observation for each security-month is retained. A monthly return is calculated only when the previous security observation is the immediately preceding month. This prevents long gaps from being treated as ordinary one-month returns.

## 6. Industry portfolio construction

The unit of analysis is the industry-month. This is essential because the research question concerns industry run-ups rather than a single German market index.

For each industry-month, the workflow computes beginning-of-month market capitalization from price, the EUR conversion factor, and shares outstanding. Industry returns are value-weighted using lagged market capitalization. A German value-weighted market portfolio is constructed from the same securities.

The net-market industry return is defined as the industry return minus the German market return. Both raw and net-market returns are used in the run-up definition.

## 7. Run-up identification

For each threshold X in 50%, 75%, 100%, 125%, and 150%, an industry qualifies when all three conditions hold:

1. Prior 24-month raw industry return is at least X.
2. Prior 24-month net-market industry return is at least X.
3. Prior 60-month raw industry return is at least 50%.

The qualifying measures are lagged one month before the event indicator is assigned. The workflow then retains distinct episodes rather than counting every qualifying month. A qualifying observation is retained only when no qualifying observation appears in the preceding 24 months. Consequently, the episode count is an event count, not a month count.

## 8. Forward outcomes, drawdowns, and crashes

The outcome window covers the 24 months after each event. An observation is considered complete only when the required monthly sequence exists through month +24.

For each future month, the workflow compares the industry wealth index with the highest value observed from event month 0 through the month before the observation being evaluated. The maximum drawdown is the most negative peak-to-trough decline over the window. A raw or net-market crash indicator equals one when the relevant maximum drawdown is at least 40% in absolute value.

This definition differs from a simple end-to-end return: an industry can finish the two-year window above its starting value and still experience a crash if it falls by 40% from an earlier peak.

## 9. Pre-run-up volatility and the regression sample

Pre-run-up annualized volatility is calculated from value-weighted daily industry returns over the preceding 12 months and annualized using the square root of 252.

The full outcome file is used for the threshold summaries and formal crash-probability comparison. Volatility is merged only for the regression specifications that require it. This prevents missing volatility from unnecessarily changing the unconditional crash-test sample.

## 10. Descriptive German run-up episodes

This exhibit lists the identified raw-only 100% German run-up episodes and summarizes their subsequent returns and drawdowns. It provides the episode-level context for the aggregate threshold results below.

The table is descriptive. It does not by itself establish that run-ups cause crashes. The main interpretive focus is whether large subsequent drawdowns coexist with strong post-run-up average returns, since these measure different features of the path.

In [ ]:
table1 = read_excel('germany_table1_paper_style.xlsx')
display(table1)
print(f'Rows: {len(table1)} | Columns: {len(table1.columns)}')


## 11. Event-time price paths

The event-time analysis follows strict 100% run-up episodes from month −24 to month +48. Each episode is normalized at event month 0. The lines distinguish the average industry path, episodes later classified as crashes, episodes not classified as crashes, and the market path.

The relevant evidence is whether crash episodes separate from non-crash episodes after the event and whether the industry path differs from the market benchmark. Because the number of strict German episodes is small, the figure is best treated as visual and descriptive evidence.

In [ ]:
event_time = read_excel('germany_event_time_paths.xlsx')
display(event_time)
print(f'Event-time rows: {len(event_time)}')
if 'event_month' in event_time.columns:
    print(f'Rows at event month 0: {(event_time["event_month"] == 0).sum()}')
display(Image(filename=str(PROJECT_DIR / 'germany_event_time_paths.png')))


## 12. Threshold-level crash and return results

The threshold exhibit summarizes the number of distinct episodes, forward raw returns, raw and net-market crash rates, and average 24-month drawdowns at each threshold.

The interpretation should look for a systematic pattern across thresholds, not just the largest percentage. Higher thresholds automatically produce fewer observations, so a high crash rate based on only a few episodes is imprecise. The German results should therefore be reported with both magnitudes and episode counts.

In [ ]:
table3 = read_excel('germany_table3_main.xlsx')
display(table3)
supporting_path = PROJECT_DIR / 'germany_crash_results.xlsx'
if supporting_path.exists():
    display(Markdown('### Detailed threshold output'))
    display(pd.read_excel(supporting_path))


## 13. Regression evidence

The regression table estimates crash indicators on raw and net-market run-up indicators at each threshold. The uncontrolled specification contains the run-up indicator alone. The controlled specification adds pre-run-up annualized volatility. Standard errors are clustered by calendar year.

The run-up coefficient is a conditional difference in crash probability within the specified linear-probability model. If the coefficient is smaller after adding volatility, that is consistent with volatility explaining part of the raw association. The small German episode sample means that coefficient size, p-values, and robustness across specifications should be considered together.

In [ ]:
regressions = read_excel('germany_footnote15_regressions.xlsx')
display(regressions)
print(f'Regression rows: {len(regressions)}')


## 14. Formal crash-probability comparison

The formal test compares the crash probability in the full complete-outcome sample with the crash probability after a strict 100% run-up. It also compares the strict 100% group with non-100% months. The table reports the estimated probabilities, differences, z statistics, and p-values.

A large numerical difference with a large p-value should be described as suggestive rather than statistically conclusive. This distinction is especially important for Germany because the strict run-up group contains few episodes.

In [ ]:
crash_test = read_excel('germany_crash_test.xlsx')
display(crash_test)
if {'Comparison', 'Observations', 'Crashes'}.issubset(crash_test.columns):
    check = crash_test[['Comparison', 'Observations', 'Crashes']].copy()
    check['Calculated probability (%)'] = check['Crashes'] / check['Observations'] * 100
    display(check)


## 15. Comparison with the U.S. evidence

The comparison should be organized around patterns rather than a mechanical equality of numerical estimates. The U.S. study contains many more industries and episodes, while the German analysis contains a smaller country-specific sample.

The relevant comparison questions are:

- Does Germany also contain large industry run-ups followed by substantial drawdowns?
- Is the crash probability after a strict 100% run-up higher than the broad unconditional rate?
- Do event-time paths separate between crash and non-crash episodes?
- Does controlling for pre-run-up volatility reduce the run-up relationship?

Evidence that points in the same direction as the U.S. results should be described as consistency with the broader pattern. Differences should be discussed in light of Germany's smaller sample, industry composition, currency history, and sample period.

## 16. Integrity checks and reproducibility

The final exhibit files are the authoritative reporting outputs. Supporting Stata files remain in the project folder so that the workflow can be inspected and rerun. This notebook does not overwrite project files.

In [ ]:
supporting_files = [
    'germany_prepared.dta',
    'germany_currency_clean.dta',
    'germany_returns_raw.dta',
    'germany_daily_clean.dta',
    'germany_monthly_security.dta',
    'germany_industry_monthly.dta',
    'germany_runup_monthly.dta',
    'germany_runup_outcomes.dta',
    'germany_industry_volatility.dta',
    'germany_gsy_analysis.dta',
]
for filename in supporting_files:
    print(f'{filename}: {"FOUND" if (PROJECT_DIR / filename).exists() else "MISSING"}')

print('Diagnostic presentation complete.')


## 17. Limitations

The main limitations are the small number of German run-up episodes, the use of one WRDS source, the fixed DEM/EUR conversion, the absence of an external risk-free series, possible differences in industry composition relative to the U.S. analysis, and the descriptive nature of the event-time figure.

The formal tests and regressions should therefore not be presented as definitive causal evidence. They provide structured evidence about whether the German data display the same broad run-up and crash pattern.

## 18. Conclusion

The German analysis applies a transparent industry-level run-up and crash framework. The final conclusion should report the actual regenerated episode counts and exhibit values, summarize the conditional versus unconditional crash comparison, describe the event-time pattern, explain the role of volatility controls, and place the German evidence alongside the broader U.S. findings.

No numbers from the earlier project description are treated as required checkpoints. The executed Stata outputs displayed in this notebook are the results used for the seminar.